# ___Decide the discrete character transition model based on the supplementary data of Maherali, H. et al. (2016)___
-------------------

In [1]:
# ‘Mutualism Persistence and Abandonment during the Evolution of the Mycorrhizal Symbiosis’, The American Naturalist, 188(5), pp. E113–E125. Available at: https://doi.org/10.1086/688675.

In [3]:
suppressPackageStartupMessages({
    library("ape")
    library("geiger")
    library("phytools")
    library("corHMM")
    library("U.PhyloMaker")
})

In [5]:
states <- read.csv("../../data/chapter2/Maherali.etal.AmNat.Data.csv") # supplementary dataset from the paper
head(states)

,Genus_species,mycorrhizal.state
,<chr>,<chr>
1,Abies_alba,EM
2,Abies_amabilis,EM
3,Abies_cephalonica,EM
4,Abies_concolor,EM
5,Abies_firma,EM
6,Abies_fraseri,EM


In [7]:
# now we need a phylogeny for all these species

megatree <- ape::read.tree("../../data/chapter2/uphylomaker/GBOTB.extended.TPL.tre") # the TPL megatree

In [15]:
# species list must be a dataframe with following columns
# species,genus,family,species.relative,genus.relative

# gsub(states$Genus_species, pattern = '_', replacement = ' ')
# gsub(states$Genus_species, pattern = "_\\w+", replacement = '')

species_list <- data.frame(species = gsub(states$Genus_species, pattern = '_', replacement = ' '),
              genus = gsub(states$Genus_species, pattern = "_\\w+", replacement = ''), family = NA, species.relative = NA, genus.relative = NA)

head(species_list) # good

,species,genus,family,species.relative,genus.relative
,<chr>,<chr>,<lgl>,<lgl>,<lgl>
1,Abies alba,Abies,NA,NA,NA
2,Abies amabilis,Abies,NA,NA,NA
3,Abies cephalonica,Abies,NA,NA,NA
4,Abies concolor,Abies,NA,NA,NA
5,Abies firma,Abies,NA,NA,NA
6,Abies fraseri,Abies,NA,NA,NA


In [19]:
unique_genera <- unique(species_list$genus)
length(unique_genera)

[1] 1337

In [20]:
# genus list needs to be a dataframe with the following columns 
# genus,family

taxonlookup <- read.csv("../../data/chapter2/plantlookup_serialized_from_r.csv")

In [23]:
mean(unique_genera %in% taxonlookup$genus) # not bad at all

[1] 0.9970082

In [ ]:
# drop the genera that do not have family info in taxonlookup


In [ ]:
hmm_er <- corHMM::corHMM(phy = phylogeny, data = data[, c("binominal", "state")], model = "ER", node.states = "marginal", rate.cat = 1)
hmm_sym <- corHMM::corHMM(phy = phylogeny, data = data[, c("binominal", "state")], model = "SYM", node.states = "marginal", rate.cat = 1)
hmm_ard <- corHMM::corHMM(phy = phylogeny, data = data[, c("binominal", "state")], model = "ARD", node.states = "marginal", rate.cat = 1)

ape_er <- ape::ace(phy = phylogeny, x = states, type = "discrete", method = "ML", model = "ER", marginal = FALSE)
ape_sym <- ape::ace(phy = phylogeny, x = states, type = "discrete", method = "ML", model = "SYM", marginal = FALSE)
ape_ard <- ape::ace(phy = phylogeny, x = states, type = "discrete", method = "ML", model = "ARD", marginal = FALSE)

ancr_er <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states, model = "ER"))
ancr_sym <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states, model = "SYM"))
ancr_ard <- phytools::ancr(phytools::fitMk(tree = phylogeny, x = states, model = "ARD"))

save(hmm_er, hmm_sym, hmm_ard, # corHMM
        ape_er, ape_sym, ape_ard, # ape
            ancr_er, ancr_sym, ancr_ard, # phytools
                file = "./../../data/chapter2/rdata/ace_states_maherali_2016.RData")